# 07 Publish the deployed model

Uploads capstone-ml/models/deployed/ with a README to the public Hugging Face repo KeittahSewe/ratings-reputation-sentiment. It then downloads every file back without a token at the pinned commit and checks it against Drive by SHA-256. Closes #37. The write token came from Colab Secrets and has since been deleted.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, os, urllib.request

DEPLOYED = '/content/drive/MyDrive/capstone-ml/models/deployed'

for name in sorted(os.listdir(DEPLOYED)):
    path = f'{DEPLOYED}/{name}'
    sha = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print(f'{name:22} {os.path.getsize(path) / 1e6:8.2f} MB  sha256 {sha[:16]}')

card = json.load(open(f'{DEPLOYED}/model_card.json'))
url = 'https://raw.githubusercontent.com/KeittahSewe/ratings-reputation-system/main/ml/model_card.json'
repo_card = json.load(urllib.request.urlopen(url))

print('\nmodel_version:', card['model_version'])
print('temperature:  ', card['temperature'])
print('labels:       ', card['labels'])
print('Drive card matches the GitHub card:', card == repo_card)

Mounted at /content/drive
model.onnx                67.36 MB  sha256 393cfaa2627857e7
model_card.json            0.00 MB  sha256 d2972421e7330e81
tokenizer.json             0.71 MB  sha256 d241a60d5e8f04cc
tokenizer_config.json      0.00 MB  sha256 74f1f913d51b6b84

model_version: distilbert-sst5-3cls-v1.0-20261111-seed43-onnx-int8
temperature:   1.2984
labels:        ['negative', 'neutral', 'positive']
Drive card matches the GitHub card: True


In [2]:
from google.colab import userdata
from huggingface_hub import whoami

print(whoami(token=userdata.get('HF_TOKEN'))['name'])

KeittahSewe


In [3]:
import shutil

STAGE = '/content/hf_upload'
shutil.rmtree(STAGE, ignore_errors=True)
shutil.copytree(DEPLOYED, STAGE)

m = card['metrics']
e = card['efficiency_colab_cpu_1_thread']
readme = f"""---
base_model: distilbert-base-uncased
language: en
pipeline_tag: text-classification
tags:
- onnx
- sentiment-analysis
---

# Ratings and reputation sentiment model

A three-class sentiment classifier (negative, neutral, positive) for reviews on online service marketplaces. It is the deployed model of a final-year BSc Informatics and Computer Science project at Strathmore University. Code and the full evaluation are at https://github.com/KeittahSewe/ratings-reputation-system.

Model version: `{card['model_version']}`

## Files

| File | What it is |
|---|---|
| model.onnx | `distilbert-base-uncased` fine-tuned on SST-5 collapsed to three classes (seed {card['seed']}), exported to ONNX and quantised to int8 |
| tokenizer.json, tokenizer_config.json | the tokenizer that goes with it |
| model_card.json | version, label order, temperature and metrics, read by the inference service at startup |

## Using it

Clean the text with `clean_text` from `ml/01_data_prep.ipynb` in the GitHub repo, then tokenise with truncation at {card['max_length']} tokens. The model takes `input_ids` and `attention_mask` and returns `logits` in the order {', '.join(card['labels'])}. Divide the logits by the temperature ({card['temperature']}) before the softmax. The label is the class with the largest probability and the confidence is that probability. Temperature scaling changes the confidence but never the label.

## Results

| Measure | Value |
|---|---|
| Marketplace test set macro-F1, this export | {m['market_macro_f1_seed_exported']} |
| Marketplace macro-F1, mean of three seeds (95% CI) | {m['market_macro_f1_three_seed_mean']} ({m['market_macro_f1_ci95'][0]} to {m['market_macro_f1_ci95'][1]}) |
| SST-5 test macro-F1, this seed | {m['sst5_test_macro_f1_seed']} |
| Expected calibration error on the marketplace set, after scaling | {m['ece_market_after_scaling']} |
| CPU latency per review, one thread (median / p95) | {e['median_ms']} ms / {e['p95_ms']} ms |
| Size on disk / peak memory while classifying | {e['size_mb']} MB / {e['peak_rss_mb']} MB |

The marketplace test set holds 390 reviews. Latency and memory were measured on Colab CPU, not on the deployment host.

## Limitations

The model was trained on movie review sentences from SST-5, not service reviews. The marketplace test set was drafted and labelled by a language model following a written guideline. A human second labeller checked 100 reviews (Cohen's kappa 0.911), and the author settled the six disagreements. Neutral is the hardest class. English only.
"""
open(f'{STAGE}/README.md', 'w').write(readme)
print(sorted(os.listdir(STAGE)))
print(readme)

['README.md', 'model.onnx', 'model_card.json', 'tokenizer.json', 'tokenizer_config.json']
---
base_model: distilbert-base-uncased
language: en
pipeline_tag: text-classification
tags:
- onnx
- sentiment-analysis
---

# Ratings and reputation sentiment model

A three-class sentiment classifier (negative, neutral, positive) for reviews on online service marketplaces. It is the deployed model of a final-year BSc Informatics and Computer Science project at Strathmore University. Code and the full evaluation are at https://github.com/KeittahSewe/ratings-reputation-system.

Model version: `distilbert-sst5-3cls-v1.0-20261111-seed43-onnx-int8`

## Files

| File | What it is |
|---|---|
| model.onnx | `distilbert-base-uncased` fine-tuned on SST-5 collapsed to three classes (seed 43), exported to ONNX and quantised to int8 |
| tokenizer.json, tokenizer_config.json | the tokenizer that goes with it |
| model_card.json | version, label order, temperature and metrics, read by the inference service a

In [4]:
from huggingface_hub import HfApi

REPO_ID = 'KeittahSewe/ratings-reputation-sentiment'
api = HfApi(token=userdata.get('HF_TOKEN'))

api.create_repo(REPO_ID, repo_type='model', private=False, exist_ok=True)
info = api.upload_folder(
    repo_id=REPO_ID,
    folder_path=STAGE,
    commit_message=f"Add {card['model_version']}",
)
print('Commit hash:', info.oid)
print(info.commit_url)

Commit hash: fd22fadb091b8795acd22c802b2552f8c3d1dab7
https://huggingface.co/KeittahSewe/ratings-reputation-sentiment/commit/fd22fadb091b8795acd22c802b2552f8c3d1dab7


In [5]:
import tempfile
from huggingface_hub import HfApi, hf_hub_download

REVISION = info.oid
public_api = HfApi(token=False)
print('Private:', public_api.model_info(REPO_ID, revision=REVISION).private)

check_dir = tempfile.mkdtemp()
all_match = True
for name in sorted(os.listdir(DEPLOYED)):
    path = hf_hub_download(REPO_ID, name, revision=REVISION, token=False, local_dir=check_dir)
    downloaded = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    original = hashlib.sha256(open(f'{DEPLOYED}/{name}', 'rb').read()).hexdigest()
    all_match &= downloaded == original
    print(f'{name:22} matches Drive: {downloaded == original}')

print('\nAll files match:', all_match)

Private: False


model.onnx: reconstructing file:   0%|          |  0.00B / 67.4MB            

model.onnx: downloading bytes:           |  0.00B            

model.onnx             matches Drive: True


model_card.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

model_card.json        matches Drive: True


tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

tokenizer.json         matches Drive: True


tokenizer_config.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer_config.json  matches Drive: True

All files match: True
